# Etude comparative de stratégies de chunking 

Le **chunking** (ou découpage de texte) est une étape clé en traitement du langage naturel (NLP) et en **RAG** (*Retrieval-Augmented Generation*). Il consiste à découper un document volumineux en sous-sections plus petites pour qu'elles puissent être analysées, vectorisées et traitées efficacement par un Modèle de Langage (LLM).

In [54]:
from dotenv import load_dotenv
load_dotenv()

with open("Bitcoin.txt", "r", encoding="utf-8") as fichier:
    text = fichier.read()

print(f"Longueur du texte : {len(text)} caractères")

Longueur du texte : 24161 caractères


## 1. Chunking par taille fixe (*Fixed-size chunking*)
C'est la méthode la plus simple. Le texte est découpé à intervalle régulier sur un nombre strict de caractères, mots ou tokens.

* Principe : Découpage strict (ex. tous les 500 caractères). Un chevauchement (*overlap*, ex. 50 caractères) est souvent ajouté pour conserver le contexte entre deux blocs adjacents.
* Inconvénient : Coupe fréquemment au milieu des mots ou des phrases, perdant la cohérence sémantique.
---



In [55]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator="",          
    chunk_size=500,        
    chunk_overlap=250,      
    length_function=len
)

chunks_fixe = text_splitter.split_text(text)
print(f"Nombre de chunks générés : {len(chunks_fixe)}")

Nombre de chunks générés : 96


## 2. Chunking récursif (*Recursive Character Splitting*)
Cette méthode s'appuie sur la structure intrinsèque du document (Markdown, HTML, JSON, Code source, PDF).

* Principe : Découpe selon les balises structurelles : titres (`#`, `##`), paragraphes (`<p>`),

---



In [69]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter_md = RecursiveCharacterTextSplitter(
    separators=[
        "\n# ",       
        "\n## ",      
        "\n### ",     
        "\n\n",       
        "\n",         
        " "           
    ],
    chunk_size=500,
    chunk_overlap=250
)

chunks_recursif = splitter_md.create_documents([text])
chunks_recursif = [chunk.page_content for chunk in chunks_recursif]
print(f"Nombre de chunks générés : {len(chunks_recursif)}")
print(f"Nombre de chunks générés : {chunks_recursif}")

Nombre de chunks générés : 98
Nombre de chunks générés : ['# Bitcoin : Un système de cash électronique de pair-à-pair\n\n**Satoshi Nakamoto**\n\nsatoshin@gmx.com\n\nwww.bitcoin.org\n\n---', '### Résumé', "Une version purement pair-à-pair d'argent électronique permettrait d'effectuer des paiements en ligne directement d'une partie à une autre, sans passer par une institution financière. Les signatures numériques fournissent une partie de la solution, mais les bénéfices majeurs sont perdus si un tiers de confiance est toujours requis pour empêcher le double dépense. Nous proposons une solution au problème de la double dépense en utilisant un réseau pair-à-pair. Le réseau horodate les transactions", 'mais les bénéfices majeurs sont perdus si un tiers de confiance est toujours requis pour empêcher le double dépense. Nous proposons une solution au problème de la double dépense en utilisant un réseau pair-à-pair. Le réseau horodate les transactions en les hachant dans une chaîne continue de 

## Token Chunking
Découpage strict basé sur le nombre de tokens calculés via un tokenizer spécifique (ex. cl100k_base pour OpenAI).

In [70]:
from langchain_text_splitters import TokenTextSplitter

text_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",  # Encodage utilisé par gpt-3.5-turbo, gpt-4 et text-embedding-3
    chunk_size=150,  # Nombre maximum de tokens par chunk
    chunk_overlap=75,  
)

# Génération des chunks de texte
chunks_token = text_splitter.split_text(text)

print(f"Nombre de chunks : {len(chunks_token)}\n")
print(f"Nombre de chunks générés : {chunks_token}")

Nombre de chunks : 89

Nombre de chunks générés : ["# Bitcoin : Un système de cash électronique de pair-à-pair\n\n**Satoshi Nakamoto**\n\nsatoshin@gmx.com\n\nwww.bitcoin.org\n\n---\n\n### Résumé\n\nUne version purement pair-à-pair d'argent électronique permettrait d'effectuer des paiements en ligne directement d'une partie à une autre, sans passer par une institution financière. Les signatures numériques fournissent une partie de la solution, mais les bénéfices majeurs sont perdus si un tiers de confiance est toujours requis pour empêcher le double dépense. Nous proposons une solution au problème de la double dépense en utilisant un réseau pair-à-pair.", ' autre, sans passer par une institution financière. Les signatures numériques fournissent une partie de la solution, mais les bénéfices majeurs sont perdus si un tiers de confiance est toujours requis pour empêcher le double dépense. Nous proposons une solution au problème de la double dépense en utilisant un réseau pair-à-pair. Le ré

## Recursive Token Chunking 
Combinaison des deux approches précédentes : il utilise la hiérarchie de séparateurs structurels (paragraphes, phrases) du découpage récursif, mais calcule la taille maximale de chaque chunk directement en tokens plutôt qu'en caractères.

In [71]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",  # Tokenizer OpenAI (text-embedding-3 / gpt-4)
    chunk_size=150,  # Taille max stricte en TOKENS
    chunk_overlap=75,
    separators=[
        "\n# ",       
        "\n## ",      
        "\n### ",     
        "\n\n",       
        "\n",         
        " "           
    ],
)

chunks_token_recursif = text_splitter.split_text(text)

print(f"Nombre de chunks : {len(chunks_token_recursif)}\n")
print(f"Nombre de chunks : {chunks_token_recursif}\n")

Nombre de chunks : 88

Nombre de chunks : ['# Bitcoin : Un système de cash électronique de pair-à-pair\n\n**Satoshi Nakamoto**\n\nsatoshin@gmx.com\n\nwww.bitcoin.org\n\n---', '### Résumé', "Une version purement pair-à-pair d'argent électronique permettrait d'effectuer des paiements en ligne directement d'une partie à une autre, sans passer par une institution financière. Les signatures numériques fournissent une partie de la solution, mais les bénéfices majeurs sont perdus si un tiers de confiance est toujours requis pour empêcher le double dépense. Nous proposons une solution au problème de la double dépense en utilisant un réseau pair-à-pair. Le réseau horodate les transactions en les hachant dans une chaîne continue de preuves de travail basées sur le hachage, formant un registre qui ne peut pas être", "toujours requis pour empêcher le double dépense. Nous proposons une solution au problème de la double dépense en utilisant un réseau pair-à-pair. Le réseau horodate les transactions 

## 4. Chunking sémantique (*Semantic Chunking*)
Une approche avancée s'appuyant sur les *embeddings* (représentations vectorielles).

* Principe : Le texte est d'abord découpé phrase par phrase. Un modèle évalue la similarité sémantique entre la phrase actuelle et la suivante. Si la variation de sens dépasse un certain seuil, une rupture de chunk est créée.
* Avantage : Chaque chunk ne contient qu'une seule idée ou thématique homogène.
* Inconvénient : Plus lent et plus coûteux en ressources de calcul.

---



In [73]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

text_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=85.0
)

chunks_semantique = text_splitter.create_documents([text])
chunks_semantique = [chunk.page_content for chunk in chunks_semantique]
print(f"Nombre de chunks générés : {len(chunks_semantique)}")
print(f"Nombre de chunks générés : {chunks_semantique}")

Nombre de chunks générés : 28
Nombre de chunks générés : ["# Bitcoin : Un système de cash électronique de pair-à-pair\n\n**Satoshi Nakamoto**\n\nsatoshin@gmx.com\n\nwww.bitcoin.org\n\n---\n\n### Résumé\n\nUne version purement pair-à-pair d'argent électronique permettrait d'effectuer des paiements en ligne directement d'une partie à une autre, sans passer par une institution financière. Les signatures numériques fournissent une partie de la solution, mais les bénéfices majeurs sont perdus si un tiers de confiance est toujours requis pour empêcher le double dépense.", "Nous proposons une solution au problème de la double dépense en utilisant un réseau pair-à-pair. Le réseau horodate les transactions en les hachant dans une chaîne continue de preuves de travail basées sur le hachage, formant un registre qui ne peut pas être modifié sans refaire la preuve de travail. La chaîne la plus longue ne sert pas seulement de preuve de la séquence des événements observés, mais aussi de preuve qu'ell

## Cluster Chunking Semantique
Le Cluster Chunking Sémantique (ou Cluster-based Semantic Chunking) est une variante avancée du chunking sémantique qui utilise un algorithme de clustering non supervisé (comme K-Means, HDBSCAN ou Agglomerative Clustering) pour regrouper les phrases d'un document en blocs thématiques homogènes, sans se limiter à l'ordre chronologique strict du texte.

## 5. Agentic / LLM-based Chunking
* Principe : On confie le texte directement à un LLM en lui demandant de déterminer lui-même où se trouvent les ruptures d'idées ou de découper le contenu selon une logique métier spécifique.
* Avantage : Qualité maximale pour des documents complexes ou très mal structurés.
* Inconvénient : Latence et coûts d'API élevés.

---

In [60]:
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class TextChunks(BaseModel):
    chunks: list[str] = Field(
        description="La liste des morceaux de texte découpés (chaînes de caractères)"
    )

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

structured_llm = llm.with_structured_output(TextChunks)

system_prompt = """Tu es un expert en traitement de texte pour systèmes RAG.
Ta mission est de découper le texte fourni en chunks logiques et autonomes.

Règles de découpage :
1. Regroupe les paragraphes qui traitent du même sujet ou de la même idée.
2. Crée un nouveau chunk dès qu'il y a un changement significatif de sujet ou de concept.
3. Ne résume pas le contenu conserve le texte source d'origine.
"""
# 4. Pour chaque chunk rempli ses metadonnées, attribue un numéro de page dans page , le nom du fichier source dans source , la section dans section et génere un identifiant unique pour le chunk.

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("human", "Voici le texte à découper :\n\n{texte}")]
)

chain = prompt | structured_llm

DocumentChunks = chain.invoke({"texte": text})

# print (f"Nombre de chunks générés : {len(DocumentChunks)}")
print (f"Exemple de chunk : {DocumentChunks.chunks[0]}")


Exemple de chunk : # Bitcoin : Un système de cash électronique de pair-à-pair

**Satoshi Nakamoto**

satoshin@gmx.com

www.bitcoin.org

---

### Résumé

Une version purement pair-à-pair d'argent électronique permettrait d'effectuer des paiements en ligne directement d'une partie à une autre, sans passer par une institution financière. Les signatures numériques fournissent une partie de la solution, mais les bénéfices majeurs sont perdus si un tiers de confiance est toujours requis pour empêcher le double dépense. Nous proposons une solution au problème de la double dépense en utilisant un réseau pair-à-pair. Le réseau horodate les transactions en les hachant dans une chaîne continue de preuves de travail basées sur le hachage, formant un registre qui ne peut pas être modifié sans refaire la preuve de travail. La chaîne la plus longue ne sert pas seulement de preuve de la séquence des événements observés, mais aussi de preuve qu'elle provient du plus grand pool de puissance CPU. Tant qu

## 6. Late Chunking (Chunking tardif)
Une technique récente développée spécifiquement pour l'indexation vectorielle.

* Principe : On passe **l'intégralité du document** dans le modèle d'embedding pour capturer le contexte global, puis on découpe les représentations vectorielles finales en sous-chunks.
* Avantage : Évite la perte du contexte global qu'un petit chunk isolé peut subir.

---

# Benchmark

In [65]:
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate, EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

template = "Réponds à la question uniquement en t'appuyant sur le contexte suivant.\n\nContexte:\n{contexte}\n\nQuestion:\n{question}"
prompt = ChatPromptTemplate.from_template(template)


def evaluate_chunking_strategy(strategy_name: str, chunks: list[str], test_questions: list[dict]):
    # 1. Indexation dans FAISS des chunks générés par la stratégie
    vectorstore = FAISS.from_texts(texts=chunks, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    rows = []

    # 2. Exécution du RAG sur le jeu de test
    for item in test_questions:
        q = item["user_input"]
        gt = item["reference"]

        retrieved_docs = retriever.invoke(q)
        retrieved_texts = [doc.page_content for doc in retrieved_docs]

        contexte_str = "\n\n".join(retrieved_texts)
        chain = prompt | llm
        response = chain.invoke({"contexte": contexte_str, "question": q})

        rows.append({
            "user_input": q,
            "response": response.content,
            "retrieved_contexts": retrieved_texts,
            "reference": gt,
        })

    # 3. Formatage pour RAGAS (nouvelle API)
    ragas_dataset = EvaluationDataset.from_list(rows)

    # 4. Calcul des métriques RAGAS
    results = evaluate(
        dataset=ragas_dataset,
        metrics=[
            LLMContextPrecisionWithReference(llm=evaluator_llm),
            LLMContextRecall(llm=evaluator_llm),
            Faithfulness(llm=evaluator_llm),
            ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
        ],
    )

    df_res = results.to_pandas()
    df_res["strategy"] = strategy_name
    return df_res

C:\Users\pierr\AppData\Local\Temp\ipykernel_15228\969097444.py:8: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\pierr\AppData\Local\Temp\ipykernel_15228\969097444.py:8: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import (
C:\Users\pierr\AppData\Local\Temp\ipykernel_15228\969097444.py:8: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\pierr\Ap

In [ ]:
# # Chunk fixe
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# # Indexation des textes dans FAISS
# vectorstore = FAISS.from_texts(
#     texts=chunks, embedding=embeddings, metadatas=metadatas
# )

# # Configuration du rôle de Retriever (extraire les 2 chunks les plus pertinents)
# retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
# # Chunk recursif
# # Chunk token
# # Chunk token recursif
# # Chunk semantique
# # Chunk llm 


In [66]:
benchmark_data = [
    # --- 1. Introduction & Contexte (Q1 à Q3) ---
    {
        "user_input": "Pourquoi le modèle bancaire traditionnel nécessite-t-il la collecte de plus d'informations personnelles ?",
        "reference": "Des transactions complètement irréversibles ne sont pas vraiment possibles, car les institutions financières ne peuvent éviter d'arbitrer les litiges... Avec la possibilité de révocation, le besoin de confiance s'étend. Les commerçants doivent se méfier de leurs clients, leur réclamant plus d'informations qu'ils n'en auraient autrement besoin.",
    },
    {
        "user_input": "Quel est le problème principal lié aux tiers de confiance pour les petites transactions ?",
        "reference": "Le coût de la médiation augmente les coûts de transaction, limitant la taille minimale pratique des transactions et coupant la possibilité de petites transactions occasionnelles.",
    },
    {
        "user_input": "Quelle est la solution proposée par Nakamoto pour remplacer la confiance dans les transactions ?",
        "reference": "Ce qui est nécessaire, c'est un système de paiement électronique basé sur la preuve cryptographique plutôt que sur la confiance, permettant à deux parties consentantes de traiter directement l'une avec l'autre sans avoir besoin d'un tiers de confiance.",
    },

    # --- 2. Transactions & Double Dépense (Q4 à Q6) ---
    {
        "user_input": "Comment Satoshi Nakamoto définit-il une pièce électronique ?",
        "reference": "Nous définissons une pièce électronique comme une chaîne de signatures numériques. Chaque propriétaire transfère la pièce au suivant en signant numériquement un hachage de la transaction précédente et la clé publique du propriétaire suivant.",
    },
    {
        "user_input": "Comment le réseau évite-t-il qu'un nœud doive télécharger tout l'historique d'une pièce ?",
        "reference": "Le seul moyen de confirmer l'absence d'une transaction est d'être informé de toutes les transactions... Pour accomplir cela sans tiers de confiance, les transactions doivent être annoncées publiquement [1], et nous avons besoin d'un système permettant aux participants de s'accorder sur un historique unique.",
    },
    {
        "user_input": "Pourquoi la solution traditionnelle de la maison de la monnaie (mint) pose-t-elle un problème de centralisation ?",
        "reference": "Le problème de cette solution est que le sort de tout le système monétaire dépend de la société qui gère la maison de la monnaie, chaque transaction devant passer par elle, tout comme une banque.",
    },

    # --- 3. Horodatage & Preuve de Travail (Q7 à Q10) ---
    {
        "user_input": "Comment fonctionne le serveur d'horodatage proposé dans le papier ?",
        "reference": "Un serveur d'horodatage fonctionne en prenant le hachage d'un bloc d'éléments à horodater et en publiant largement ce hachage... Chaque horodatage inclut l'horodatage précédent dans son hachage, formant une chaîne.",
    },
    {
        "user_input": "Sur quel algorithme de preuve de travail le réseau Bitcoin s'appuie-t-il ?",
        "reference": "Pour mettre en œuvre un serveur d'horodatage distribué en pair-à-pair, nous devrons utiliser un système de preuve de travail similaire à Hashcash d'Adam Back [6].",
    },
    {
        "user_input": "Pourquoi la preuve de travail est-elle définie comme un CPU = une voix ?",
        "reference": "Si la majorité était basée sur une adresse IP = une voix, elle pourrait être subvertie par quiconque capable d'allouer de nombreuses adresses IP. La preuve de travail est essentiellement un CPU = une voix.",
    },
    {
        "user_input": "Comment la difficulté de la preuve de travail s'ajuste-t-elle dans le temps ?",
        "reference": "La difficulté de la preuve de travail est déterminée par une moyenne mobile ciblant un nombre moyen de blocs par heure. S'ils sont générés trop vite, la difficulté augmente.",
    },

    # --- 4. Fonctionnement du Réseau (Q11 à Q13) ---
    {
        "user_input": "Quelles sont les 6 étapes pour exécuter le réseau Bitcoin ?",
        "reference": "1. Les nouvelles transactions sont diffusées à tous les nœuds. 2. Chaque nœud collecte les nouvelles transactions dans un bloc. 3. Chaque nœud travaille à trouver une preuve de travail difficile pour son bloc. 4. Lorsqu'un nœud trouve une preuve de travail, il diffuse le bloc à tous les nœuds. 5. Les nœuds n'acceptent le bloc que si toutes les transactions qu'il contient sont valides. 6. Les nœuds expriment leur acceptation du bloc en travaillant sur la création du bloc suivant.",
    },
    {
        "user_input": "Que font les nœuds si deux versions différentes du bloc suivant sont diffusées en même temps ?",
        "reference": "Ils travaillent sur le premier reçu, mais conservent l'autre branche au cas où elle deviendrait plus longue. L'égalité sera rompue lorsque la preuve de travail suivante sera trouvée et qu'une branche deviendra plus longue.",
    },
    {
        "user_input": "Comment le réseau réagit-il si une transaction ou un bloc manque un nœud lors de la diffusion ?",
        "reference": "Les diffusions de nouvelles transactions n'ont pas nécessairement besoin d'atteindre tous les nœuds... Si un nœud ne reçoit pas un bloc, il le demandera lorsqu'il recevra le bloc suivant et réalisera qu'il en a manqué un.",
    },

    # --- 5. Incitations Économiques (Q14 à Q16) ---
    {
        "user_input": "De quelles deux manières les mineurs sont-ils rémunérés ?",
        "reference": "La première transaction d'un bloc est une transaction spéciale qui démarre une nouvelle pièce possédée par le créateur du bloc... L'incitation peut également être financée par des frais de transaction. Si la valeur de sortie d'une transaction est inférieure à sa valeur d'entrée, la différence est un frais.",
    },
    {
        "user_input": "Que se passera-t-il pour la rémunération des nœuds une fois qu'un nombre prédéterminé de pièces sera en circulation ?",
        "reference": "Une fois qu'un nombre prédéterminé de pièces sera entré en circulation, l'incitation pourra transitionner entièrement vers les frais de transaction et être complètement exempte d'inflation.",
    },
    {
        "user_input": "Pourquoi un attaquant possédant une majorité de puissance CPU trouverait-il plus profitable de rester honnête ?",
        "reference": "Il devrait trouver plus profitable de jouer selon les règles (des règles qui lui favorisent plus de nouvelles pièces que tous les autres combinés) plutôt que de saper le système et la validité de sa propre richesse.",
    },

    # --- 6. Optimisation Mémoire & SPV (Q17 à Q20) ---
    {
        "user_input": "Quel est le rôle de l'arbre de Merkle dans la gestion de l'espace disque des nœuds ?",
        "reference": "Une fois que la dernière transaction d'une pièce est enterrée sous suffisamment de blocs, les transactions dépensées avant elle peuvent être jetées pour économiser de l'espace disque... les transactions sont hachées dans un arbre de Merkle, avec seule la racine incluse dans le hachage du bloc.",
    },
    {
        "user_input": "Quelle est la taille approximative d'un en-tête de bloc sans transactions et quel est son impact annuel ?",
        "reference": "Un en-tête de bloc sans transactions ferait environ 80 octets. Si nous supposons que les blocs sont générés toutes les 10 minutes, 80 octets x 6 x 24 x 365 = 4,2 Mo par an.",
    },
    {
        "user_input": "Comment fonctionne la vérification simplifiée des paiements (SPV) pour un nœud léger ?",
        "reference": "Un utilisateur a seulement besoin de garder une copie des en-têtes de bloc de la chaîne de preuve de travail la plus longue... et d'obtenir la branche de Merkle liant la transaction au bloc dans lequel elle est horodatée.",
    },
    {
        "user_input": "Comment un logiciel léger SPV peut-il se protéger contre un bloc invalide créé par un attaquant ?",
        "reference": "Une stratégie pour se protéger contre cela serait d'accepter les alertes des nœuds du réseau lorsqu'ils détectent un bloc invalide, invitant le logiciel de l'utilisateur à télécharger le bloc complet.",
    },

    # --- 7. Transactions Avancées & Confidentialité (Q21 à Q23) ---
    {
        "user_input": "Comment les transactions Bitcoin gèrent-elles la division et la combinaison des valeurs ?",
        "reference": "Pour permettre à la valeur d'être divisée et combinée, les transactions contiennent des entrées (inputs) et des sorties (outputs) multiples.",
    },
    {
        "user_input": "Quel est le risque de confidentialité lié aux transactions à plusieurs entrées ?",
        "reference": "Un certain liage est toujours inévitable avec les transactions à plusieurs entrées, qui révèlent nécessairement que leurs entrées appartenaient au même propriétaire.",
    },
    {
        "user_input": "Comment la confidentialité est-elle préservée dans Bitcoin alors que toutes les transactions sont publiques ?",
        "reference": "La confidentialité peut toujours être maintenue en brisant le flux d'informations à un autre endroit : en gardant les clés publiques anonymes... Comme pare-feu supplémentaire, une nouvelle paire de clés devrait être utilisée pour chaque transaction.",
    },

    # --- 8. Calculs de Sécurité & Attaques (Q24 à Q28) ---
    {
        "user_input": "Quelles sont les limites d'un attaquant s'il parvient à générer une chaîne alternative plus rapide que la chaîne honnête ?",
        "reference": "Cela n'ouvre pas le système à des changements arbitraires, comme créer de la valeur à partir de rien ou prendre de l'argent qui n'a jamais appartenu à l'attaquant... Un attaquant peut seulement essayer de modifier l'une de ses propres transactions pour reprendre l'argent qu'il a récemment dépensé.",
    },
    {
        "user_input": "À quel problème classique des probabilités la course entre la chaîne honnête et la chaîne de l'attaquant est-elle comparée ?",
        "reference": "La course entre la chaîne honnête et une chaîne d'attaquant peut être caractérisée comme une Marche Aléatoire Binomiale... La probabilité qu'un attaquant rattrape un déficit donné est analogue au problème de la Ruine du Joueur (Gambler's Ruin).",
    },
    {
        "user_input": "Quelle est la formule mathématique de la probabilité q_z qu'un attaquant rattrape un retard de z blocs si p > q ?",
        "reference": "q_z = (q/p)^z si p > q",
    },
    {
        "user_input": "Quelle distribution statistique est utilisée pour modéliser le progrès potentiel d'un attaquant au fil du temps ?",
        "reference": "Le progrès potentiel de l'attaquant sera une distribution de Poisson avec une valeur attendue : lambda = z * (q / p)",
    },
    {
        "user_input": "Combien de confirmations z faut-il attendre pour maintenir la probabilité de succès d'un attaquant sous 0,001 si son ratio de puissance q est de 0,10 ?",
        "reference": "P < 0.001 / q=0.10 z=5",
    },

    # --- 9. Conclusion & Références (Q29 à Q30) ---
    {
        "user_input": "Comment les nœuds expriment-ils leur vote pour accepter ou rejeter les règles du réseau ?",
        "reference": "Ils votent avec leur puissance CPU, exprimant leur acceptation des blocs valides en travaillant à les étendre et rejetant les blocs invalides en refusant de travailler dessus.",
    },
    {
        "user_input": "Quels travaux antérieurs sur l'horodatage numérique et la monnaie B-money sont cités dans les références du papier ?",
        "reference": "W. Dai, 'b-money' (1998) et S. Haber, W.S. Stornetta, 'How to time-stamp a digital document' (1991).",
    },
]

In [74]:
# Exécution des évaluations
df_fixe = evaluate_chunking_strategy("Fixe Chunking", chunks_fixe, benchmark_data)
df_recursive = evaluate_chunking_strategy("Recursive Chunking", chunks_recursif, benchmark_data)
df_token = evaluate_chunking_strategy("Token Chunking", chunks_token, benchmark_data)
df_recursive_token = evaluate_chunking_strategy("Recursive Token", chunks_token_recursif, benchmark_data)
df_semantic = evaluate_chunking_strategy("Semantic Chunking", chunks_semantique, benchmark_data)
df_llm = evaluate_chunking_strategy("LLM Chunking", DocumentChunks.chunks, benchmark_data)

# Consolidation
df_all = pd.concat(
    [df_fixe, df_recursive, df_token, df_recursive_token, df_semantic, df_llm],
    ignore_index=True,
)

# Vérifie les noms réels des colonnes de métriques avant de les figer en dur
print(df_all.columns.tolist())

metric_cols = [c for c in df_all.columns if c not in ("user_input", "response", "retrieved_contexts", "reference", "strategy")]

summary = df_all.groupby("strategy")[metric_cols].mean().reset_index()

print("=== TABLEAU COMPARATIF RAGAS ===")
print(summary)

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

Exception raised in Job[19]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-viPB18Xd1pULmYl3fXnAQle2 on tokens per min (TPM): Limit 200000, Used 200000, Requested 492. Please try again in 147ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[40]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-viPB18Xd1pULmYl3fXnAQle2 on tokens per min (TPM): Limit 200000, Used 200000, Requested 7424. Please try again in 2.227s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
LLM returned 1 generations instead of requ

['user_input', 'retrieved_contexts', 'response', 'reference', 'llm_context_precision_with_reference', 'context_recall', 'faithfulness', 'answer_relevancy', 'strategy']
=== TABLEAU COMPARATIF RAGAS ===
             strategy  llm_context_precision_with_reference  context_recall  \
0       Fixe Chunking                              0.886111        0.833333   
1        LLM Chunking                              1.000000        1.000000   
2  Recursive Chunking                              0.930556        0.855556   
3     Recursive Token                              0.872222        0.855556   
4   Semantic Chunking                              0.933333        0.905556   
5      Token Chunking                              0.916667        0.855556   

   faithfulness  answer_relevancy  
0      0.846687          0.859156  
1      0.898148          0.896075  
2      0.767476          0.856449  
3      0.788896          0.836199  
4      0.740355          0.868078  
5      0.830462          0.88